In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [3]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [4]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [5]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [6]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [9]:
square_root.invoke({"x": 9})

3.0

In [11]:
get_weather.invoke({"location": "广州", "include_forecast": True})

'Current weather in 广州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [12]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 创建智能体
llm = ChatOpenAI(
    model="qwen3.7-max-2026-06-08",
    base_url=base_url,
    api_key=api_key,
    extra_body={"enable_thinking": False},  # 关闭思考模式
)

agent = create_agent(
    model=llm,
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

In [13]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="广州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Current weather in 广州: 22 degrees C
Next 5 days: Sunny广州当前气温为22摄氏度，接下来5天的天气预报为晴天。

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="99999和11111的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

99999和11111的平方根是多少?
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_131bb792361544ae8fc476d0)
 Call ID: call_131bb792361544ae8fc476d0
  Args:
    x: 99999
  square_root (call_8f2e0104f3b34f5d8e60ca3a)
 Call ID: call_8f2e0104f3b34f5d8e60ca3a
  Args:
    x: 11111
None
================================= Tool Message =================================
Name: square_root

316.226184874055
None
================================= Tool Message =================================
Name: square_root

105.40872829135166
None
================================== Ai Message ==================================

99999的平方根约为316.23，11111的平方根约为105.41。
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [16]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=2,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [17]:
search_tool.invoke("蒸蚌是什么梗？")

{'error': ValueError("Error 432: This request exceeds your plan's set usage limit. Please upgrade your plan or contact support@tavily.com")}

In [13]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [14]:
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================

我来帮你搜索一下“蒸蚌”这个梗的含义和来源。
Tool Calls:
  tavily_search (call_00_dxRu6TxC9ny4s2V9SuManoVc)
 Call ID: call_00_dxRu6TxC9ny4s2V9SuManoVc
  Args:
    query: 蒸蚌 是什么梗 网络用语 含义 来源
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 是什么梗 网络用语 含义 来源", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://hinative.com/questions/15904858", "title": "What is the meaning of \"我蒸蚌\"? - HiNative", "content": "蒸蚌是「真棒」的意思因為“蒸蚌” 跟“真棒” 的音一模一樣，所以有一些人在寫「真棒」的時候，會故意用蒸蚌兩字，然後只是因為這樣子看起來很有趣 . 蒸", "score": 0.9997063, "raw_content": null}, {"url": "https://www.douyin.com/search/%E8%92%B8%E8%9A%8C%E6%A2%97%E5%87%BA%E5%A4%84", "title": "蒸蚌梗出处- 抖音 - Douyin", "content": "00:01 \"周深蒸蚌\"：粉丝玩梗的欢乐风暴。\"周深蒸蚌\"并非新歌，而是粉丝圈里火出圈的玩梗口号。它起源于周深在短视频中频繁使用的口头禅\"蒸蚌\"，凭借超强的", "s

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [15]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [16]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [17]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [18]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='"蒸蚌"是一个网络流行梗，主要有两层含义：\n\n1. **谐音梗**："蒸蚌"是"真棒"的谐音，因为发音完全相同，所以网友故意用"蒸蚌"这两个字来代替"真棒"，显得有趣好玩。\n\n2. **源自"萝卜纸巾猫"梗**：这个梗主要来源于抖音博主"超级无敌大开门"的视频。博主训练他的三花猫"开门"辨认物品，当猫咪在胡萝卜和纸巾之间选择时，无论选对选错，主人都会用魔性的声音说"真棒"（谐音"蒸蚌"）来鼓励猫咪。\n\n**梗的流行原因**：\n- 猫咪"开门"在辨认胡萝卜和纸巾时，总是"察言观色"地蒙题\n- 主人那句魔性的"蒸蚌"夸赞声极具感染力\n- 猫咪的可爱表现和主人的互动形成了反差萌\n- 谐音梗本身具有记忆点和传播力\n\n**发展**：\n这个梗从猫咪视频逐渐演变成全网模仿的热潮，出现了各种"蒸蚌"挑战和模仿，甚至出现了"猫传人"现象，成为2025年的一个爆款网络梗。现在"蒸蚌"已经成为一种幽默、自我调侃式的鼓励文化符号。' reference=[Reference(title='蒸蚌梗源自胡萝卜纸巾猫_新浪新闻', url='https://www.sina.cn/news/detail/5260814380434515.html'), Reference(title='蒸蚌小猫梗怎么火的 - 抖音', url='https://www.douyin.com/shipin/7589558380507842611'), Reference(title='"我蒸蚌"是什么意思？ - HiNative', url='https://zh.hinative.com/questions/15904858'), Reference(title='萝卜纸巾分不清的"蒸蚌小猫"，拿下首个全球代言！ - 新浪财经', url='https://cj.sina.cn/articles/view/5952915705/162d248f906702n1fe?froms=ggmp&vt=4')]
